# Pendekatan Klasik

In [ ]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import re

## Load Dataset

In [ ]:
# === 1. Mount Google Drive ===
from google.colab import drive
drive.mount('/content/drive')
data_path = "/content/drive/MyDrive/Label_berita_New.csv"

# Baca dataset
df = pd.read_csv(data_path)

# Lihat daftar kolom untuk memastikan nama kolom benar
print("Kolom dalam dataset:", df.columns.tolist())

# Pilih kolom yang digunakan (ubah sesuai nama kolom dataset kamu)
df = df[['judul_berita', 'label']].dropna()

# Tampilkan 5 data pertama
df.head()

# Cek jumlah data per sentimen
print("\nJumlah data per sentimen:")
print(df['label'].value_counts())

# Persentase masing-masing sentimen
print("\nPersentase tiap sentimen:")
print(df['label'].value_counts(normalize=True) * 100)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Kolom dalam dataset: ['judul_berita', 'isi', 'date', 'url', 'label']

Jumlah data per sentimen:
label
Netral     2336
Positif    2223
Negatif     741
Name: count, dtype: int64

Persentase tiap sentimen:
label
Netral     44.075472
Positif    41.943396
Negatif    13.981132
Name: proportion, dtype: float64


## Preprocessing Teks Bahasa Indonesia

In [ ]:
import re
import pandas as pd
from tqdm.notebook import tqdm
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# === Aktifkan progress bar ===
tqdm.pandas()

# Inisialisasi stemmer & stopword remover
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stop_factory = StopWordRemoverFactory()
stopwords = set(stop_factory.get_stop_words())

# === Pilihan Cepat / Lengkap ===
USE_STEMMING = False  # ubah ke True jika ingin pakai stemming penuh (lebih lama)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)           # hapus simbol, angka
    tokens = text.split()
    tokens = [w for w in tokens if w not in stopwords]
    text = " ".join(tokens)


    if USE_STEMMING:
        text = stemmer.stem(text)                  # aktifkan hanya jika perlu
    return text

## A. TF-IDF Vectorization

In [ ]:
# === Buat kolom clean_text dulu ===
df['clean_text'] = df['judul_berita'].progress_apply(clean_text)

# Pisahkan fitur (X) dan label (y)
X = df['clean_text']
y = df['label']

# Bagi data untuk training dan testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Konversi teks menjadi vektor TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Jumlah fitur TF-IDF:", len(vectorizer.get_feature_names_out()))

  0%|          | 0/5300 [00:00<?, ?it/s]

Jumlah fitur TF-IDF: 5000


## Naive Bayes

In [ ]:
# Buat dan latih model
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

# Prediksi data test
y_pred = nb.predict(X_test_tfidf)

In [ ]:
print("=== HASIL EVALUASI MODEL KLASIK ===")
print("Akurasi :", round(accuracy_score(y_test, y_pred)*100, 2), "%\n")

print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

=== HASIL EVALUASI MODEL KLASIK ===
Akurasi : 53.87 %

=== Classification Report ===
              precision    recall  f1-score   support

     Negatif       0.69      0.06      0.11       147
      Netral       0.51      0.61      0.55       462
     Positif       0.57      0.62      0.59       451

    accuracy                           0.54      1060
   macro avg       0.59      0.43      0.42      1060
weighted avg       0.56      0.54      0.51      1060

=== Confusion Matrix ===
[[  9 103  35]
 [  4 282 176]
 [  0 171 280]]


## Model SVM

In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train_tfidf, y_train)
y_pred_svm = svm.predict(X_test_tfidf)

In [ ]:
print("\n=== Support Vector Machine ===")
print("Akurasi:", round(accuracy_score(y_test, y_pred_svm)*100, 2), "%\n")
print(confusion_matrix(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))


=== Support Vector Machine ===
Akurasi: 58.11 %

[[ 56  55  36]
 [ 28 286 148]
 [ 19 158 274]]
              precision    recall  f1-score   support

     Negatif       0.54      0.38      0.45       147
      Netral       0.57      0.62      0.60       462
     Positif       0.60      0.61      0.60       451

    accuracy                           0.58      1060
   macro avg       0.57      0.54      0.55      1060
weighted avg       0.58      0.58      0.58      1060



## rendome Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_tfidf, y_train)
y_pred_rf = rf.predict(X_test_tfidf)

In [ ]:
print("\n=== Random Forest ===")
print("Akurasi:", round(accuracy_score(y_test, y_pred_rf)*100, 2), "%\n")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


=== Random Forest ===
Akurasi: 58.49 %

[[ 42  65  40]
 [ 11 307 144]
 [  3 177 271]]
              precision    recall  f1-score   support

     Negatif       0.75      0.29      0.41       147
      Netral       0.56      0.66      0.61       462
     Positif       0.60      0.60      0.60       451

    accuracy                           0.58      1060
   macro avg       0.63      0.52      0.54      1060
weighted avg       0.60      0.58      0.58      1060



##B. BoW

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(max_features=5000, ngram_range=(1,2))
X_bow = vectorizer.fit_transform(X)

In [ ]:
X_train_bow, X_test_bow, y_train_bow, y_test_bow = train_test_split(
    X_bow, y, test_size=0.2, random_state=42
)

## BoW + Naive Bayes

In [ ]:
nb = MultinomialNB()
nb.fit(X_train_bow, y_train_bow)
y_pred_nb = nb.predict(X_test_bow)

print("Akurasi NB (BoW):", accuracy_score(y_test_bow, y_pred_nb))
print(classification_report(y_test_bow, y_pred_nb))

Akurasi NB (BoW): 0.5443396226415095
              precision    recall  f1-score   support

     Negatif       0.46      0.40      0.43       147
      Netral       0.54      0.52      0.53       462
     Positif       0.57      0.62      0.59       451

    accuracy                           0.54      1060
   macro avg       0.52      0.51      0.52      1060
weighted avg       0.54      0.54      0.54      1060



## BoW + SVM

In [ ]:
svm = LinearSVC()
svm.fit(X_train_bow, y_train_bow)

y_pred_svm_bow = svm.predict(X_test_bow)

print("Akurasi SVM (BoW):", accuracy_score(y_test_bow, y_pred_svm_bow))
print(classification_report(y_test_bow, y_pred_svm_bow))

Akurasi SVM (BoW): 0.5764150943396227
              precision    recall  f1-score   support

     Negatif       0.48      0.43      0.45       147
      Netral       0.59      0.60      0.60       462
     Positif       0.59      0.60      0.59       451

    accuracy                           0.58      1060
   macro avg       0.55      0.54      0.55      1060
weighted avg       0.57      0.58      0.58      1060

